# Problem Statement 5: Đánh giá hiệu quả chuyển đổi bán hàng và chất lượng giao dịch của nhân viên

**Mục tiêu:** đánh giá từng nhân viên (`sales_employee_id`) về khả năng chuyển đổi cơ hội bán hàng thành giao dịch thành công và chất lượng xử lý đơn hàng.

**Bảng dữ liệu dùng:** `ORDER` (`order_id`, `sales_employee_id`, `order_status`) và `ORDER_ITEMS` (`quantity`, `unit_price`, `discount_amount`).

## Định nghĩa các chỉ số
| Chỉ số | Định nghĩa |
|---|---|
| Hóa đơn được xử lý | Mọi đơn của nhân viên trong bảng `ORDER` (`total_invoices`) |
| Hóa đơn thành công | Đơn có `order_status = 'delivered'` |
| Tỷ lệ chuyển đổi | `successful_invoices / total_invoices * 100` |
| Giá trị hóa đơn | `SUM(quantity * unit_price - discount_amount)` theo `order_id` |
| Doanh thu (`total_sales`) | Tổng giá trị các hóa đơn thành công (`delivered`), khớp cách tính của PS1 |
| Giá trị hóa đơn trung bình | `total_sales / successful_invoices` |
| Chất lượng giao dịch | `100 - (returned + cancelled) / total_invoices * 100` (càng cao càng tốt) |

**Lưu ý:** ngoài `delivered`, `returned`, `cancelled` còn các trạng thái khác (created, paid, shipped...), nên 3 nhóm trên không cộng lại bằng tổng số hóa đơn.

## 1. Đọc dữ liệu

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

orders = pd.read_csv("ORDER.csv", encoding="utf-8-sig")
order_items = pd.read_csv("ORDER_ITEMS.csv", encoding="utf-8-sig")

print(orders["order_status"].value_counts())

## 2. Giá trị từng hóa đơn
Giá trị dòng hàng = số lượng × đơn giá − giảm giá, sau đó cộng theo `order_id`.

In [ ]:
order_items["line_revenue"] = (
    order_items["quantity"] * order_items["unit_price"]
    - order_items["discount_amount"]
)

invoice_value = (
    order_items.groupby("order_id", as_index=False)["line_revenue"]
    .sum()
    .rename(columns={"line_revenue": "invoice_value"})
)

data = orders.merge(invoice_value, on="order_id", how="left")
data["invoice_value"] = data["invoice_value"].fillna(0)
data["successful_invoice"] = (data["order_status"] == "delivered").astype(int)
# Doanh thu chỉ tính trên đơn thành công (khớp PS1)
data["delivered_value"] = data["invoice_value"] * data["successful_invoice"]

## 3. Thống kê theo nhân viên (yêu cầu 1 và 2)

In [ ]:
employee_analysis = (
    data.groupby("sales_employee_id")
    .agg(
        total_invoices=("order_id", "nunique"),
        successful_invoices=("successful_invoice", "sum"),
        total_sales=("delivered_value", "sum"),
        total_sales_all_orders=("invoice_value", "sum"),
        returned_invoices=("order_status", lambda x: (x == "returned").sum()),
        cancelled_invoices=("order_status", lambda x: (x == "cancelled").sum()),
    )
    .reset_index()
)

employee_analysis["average_sales_per_invoice"] = (
    employee_analysis["total_sales"] / employee_analysis["successful_invoices"]
)

employee_analysis["conversion_rate"] = (
    employee_analysis["successful_invoices"] / employee_analysis["total_invoices"] * 100
)

# Chất lượng giao dịch: tỷ lệ đơn KHÔNG bị hoàn trả hoặc hủy
employee_analysis["transaction_quality"] = 100 - (
    (employee_analysis["returned_invoices"] + employee_analysis["cancelled_invoices"])
    / employee_analysis["total_invoices"] * 100
)

## 4. So sánh số lượng hóa đơn và tổng doanh thu (yêu cầu 3)
Biểu đồ phân tán: nhân viên nằm phía trên đường xu hướng có doanh thu cao hơn mức kỳ vọng theo số hóa đơn, tức tạo giá trị cao trên mỗi giao dịch.

In [ ]:
import numpy as np

x = employee_analysis["total_invoices"]
y = employee_analysis["total_sales"]
slope, intercept = np.polyfit(x, y, 1)

employee_analysis["sales_vs_trend"] = y - (slope * x + intercept)   # > 0: doanh thu vượt xu hướng
employee_analysis["rank_total_sales"] = y.rank(method="min", ascending=False).astype(int)

plt.figure(figsize=(8, 6))
plt.scatter(x, y, alpha=0.6)
plt.plot(np.sort(x), slope * np.sort(x) + intercept, color="red", label="Xu hướng")
plt.xlabel("Số hóa đơn xử lý")
plt.ylabel("Tổng doanh thu")
plt.title("Số hóa đơn so với tổng doanh thu của từng nhân viên")
plt.legend()
plt.tight_layout()
plt.savefig("ps5_invoices_vs_sales_delivered.png", dpi=150)
plt.show()

print("Tương quan số hóa đơn - doanh thu:", round(x.corr(y), 3))
print("Tương quan số hóa đơn - doanh thu TB/hóa đơn:",
      round(x.corr(employee_analysis["average_sales_per_invoice"]), 3))

## 5. Xếp hạng nhân viên (yêu cầu 4)
Xếp hạng theo 3 tiêu chí: tỷ lệ chuyển đổi, giá trị hóa đơn trung bình, chất lượng giao dịch (hạng 1 là tốt nhất). Điểm tổng hợp là trung bình 3 hạng.

In [ ]:
for col, rank_col in [
    ("conversion_rate", "rank_conversion"),
    ("average_sales_per_invoice", "rank_average_invoice"),
    ("transaction_quality", "rank_transaction_quality"),
]:
    employee_analysis[rank_col] = (
        employee_analysis[col].rank(method="min", ascending=False).astype(int)
    )

employee_analysis["overall_score"] = employee_analysis[
    ["rank_conversion", "rank_average_invoice", "rank_transaction_quality"]
].mean(axis=1)
employee_analysis["overall_rank"] = (
    employee_analysis["overall_score"].rank(method="min").astype(int)
)

employee_analysis = employee_analysis.sort_values("overall_rank")

## 6. Đề xuất chính sách
Chênh lệch giữa các nhân viên khá nhỏ (xem hệ số biến thiên bên dưới), nên **không chia ép 25% mỗi nhóm**. Chỉ tách 10% đầu và 10% cuối theo điểm tổng hợp, phần còn lại giữ nguyên. Điểm đáng chú ý hơn là nhân viên mạnh ở chỉ số này nhưng yếu ở chỉ số kia.

In [ ]:
n = len(employee_analysis)
top_cut = employee_analysis["overall_rank"].quantile(0.10)
bottom_cut = employee_analysis["overall_rank"].quantile(0.90)

def policy(row):
    if row["overall_rank"] <= top_cut:
        return "Khen thưởng"
    if row["overall_rank"] >= bottom_cut:
        return "Đào tạo / cải thiện"
    return "Duy trì"

employee_analysis["recommendation"] = employee_analysis.apply(policy, axis=1)

# Nhân viên có giá trị hóa đơn cao nhưng chuyển đổi thấp (và ngược lại) -> cần coaching riêng
employee_analysis["imbalance"] = (
    employee_analysis["rank_conversion"] - employee_analysis["rank_average_invoice"]
)

cv = employee_analysis[["conversion_rate", "average_sales_per_invoice",
                        "transaction_quality", "total_sales"]].std() / \
     employee_analysis[["conversion_rate", "average_sales_per_invoice",
                        "transaction_quality", "total_sales"]].mean() * 100
print("Hệ số biến thiên (%):")
print(cv.round(2))

print(employee_analysis["recommendation"].value_counts())

## 7. Kết quả và lưu file

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print(employee_analysis.head(10).to_string(index=False))
print("...")
print(employee_analysis.tail(10).to_string(index=False))

employee_analysis.to_csv("ps5_employee_analysis_delivered.csv", index=False, encoding="utf-8-sig")
print("Đã lưu: ps5_employee_analysis_delivered.csv")

## 8. Nhận xét
- **Giả định:** hóa đơn thành công là đơn `delivered`. Doanh thu (`total_sales`) chỉ tính trên đơn `delivered`, khớp PS1; cột `total_sales_all_orders` giữ doanh thu trên mọi đơn để tham khảo.
- **Chênh lệch nhỏ:** các chỉ số giữa 200 nhân viên chỉ chênh vài phần trăm, nên thứ hạng nhạy với dao động nhỏ và cần thận trọng khi dùng để thưởng phạt.
- **Tương quan:** doanh thu phụ thuộc mạnh vào số hóa đơn, còn giá trị hóa đơn trung bình gần như độc lập với số hóa đơn. Vì vậy cần xem cả hai, không chỉ doanh thu.
- **Chuyển đổi và chất lượng** đều dựa vào tỷ lệ đơn không thành công nên có tương quan khá cao, nhưng không còn trùng hoàn toàn.
- **Đề xuất:** khen thưởng nhóm đầu 10%; đào tạo nhóm cuối 10%; với nhân viên lệch (`imbalance` lớn) thì coaching đúng điểm yếu (chốt đơn hoặc nâng giá trị đơn).